## SEED

In [1]:
"""
Strict Inductive Spatio-Temporal MAE (STMAE) + MSC-TimesNet Pipeline (SEED 3-Class)
===================================================================================
Specifications:
  1. Fast Reconstruction STMAE: Precomputed GPU Mask Bank + Non-overlapping Pretraining.
  2. 100% IEEE Strict Inductive Validity: Pretraining executed strictly on 14 train subjects.
  3. Context Receptive Field: 40s (T=10 windows, 4s per step).
  4. Temporal MSC-TimesNet: Top-2 2D periodic multi-scale convolutions.
  5. Metrics: Window-level & Trial-level (Acc, Balanced Acc, Macro-F1, Weighted-F1).
  6. Channel Attribution: Standalone sensitivities & cumulative Top-K channel subsets.
"""

from collections import defaultdict
import copy
import math
import os
import random
import re
import time
import warnings

import numpy as np
import pandas as pd
import scipy.io as sio
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# 1. CONFIGURATION & ACCELERATION HYPERPARAMETERS
# =====================================================================
RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CHANNELS = 62
NUM_CLASSES = 3            # 0: Negative, 1: Neutral, 2: Positive
RAW_DIM = 10               # 5 log-PSD + 5 DE
FS = 200                   # Sampling rate (Hz)
EPOCH_SEC = 4              # 4s per frame
WINDOW_LENGTH = 10         # T=10 frames (40s receptive field)
STRIDE = 1
TRIALS_PER_SESSION = 15

# Fast STMAE Pretraining Parameters (~35s per fold with AMP + Batch Size 256)
FOLD_PRETRAIN_EPOCHS = 50
PRETRAIN_BATCH_SIZE = 256
PRETRAIN_LR = 1e-3
PRETRAIN_STRIDE = 10       # Non-overlapping windows exclusively for fast pretraining

# Downstream Supervised Fine-Tuning Parameters
CLASSIFIER_EPOCHS = 25
CLASSIFIER_BATCH_SIZE = 128
CLASSIFIER_LR = 1e-3
ENCODER_LR_SCALE = 0.1     # Discriminative LR for pretrained spatial STMAE
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
CHANNEL_DROPOUT_RATE = 0.1

DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/yunzinan/seed-preprocessed/Preprocessed_EEG",
    "/kaggle/input/seed-preprocessed/Preprocessed_EEG",
    "/kaggle/input/datasets/yunzinan/seed-preprocessed",
    "/kaggle/input/seed-preprocessed",
    "./Preprocessed_EEG",
]
CACHE_FILE = "/kaggle/working/seed_40s_de_psd_cache.npz"
OUTPUT_DIR = "/kaggle/working/seed_strict_inductive_fast_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FREQUENCY_BANDS = {
    "delta": (1.0, 3.0),
    "theta": (4.0, 7.0),
    "alpha": (8.0, 13.0),
    "beta":  (14.0, 30.0),
    "gamma": (31.0, 50.0)
}

SEED_TRIAL_LABELS = [1, 0, 2, 2, 1, 0, 1, 2, 0, 0, 1, 2, 0, 2, 1]

CHANNEL_NAMES = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ",
    "F2", "F4", "F6", "F8", "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2",
    "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4",
    "C6", "T8", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ",
    "O2", "CB2"
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS


def seed_everything(seed=RANDOM_SEED):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)


# =====================================================================
# 2. 2D SCALP TOPOLOGY & FAST GPU MASK BANK (PHY-10)
# =====================================================================
def build_scalp_coords():
  rows = [
      (["FP1", "FPZ", "FP2"], 0.95),
      (["AF3", "AF4"], 0.80),
      (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
      (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
      (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
      (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
      (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
      (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
      (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
  ]
  coords = {}
  for names, y in rows:
    n = len(names)
    xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
    for name, x in zip(names, xs):
      coords[name] = (float(x), float(y))
  return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_phy_10_regions():
  coords = build_scalp_coords()
  regions = defaultdict(list)
  for idx, (x, y) in enumerate(coords):
    if y >= 0.70:
      reg = "prefrontal"
    elif y >= 0.35 and abs(x) <= 0.45:
      reg = "frontal_mid"
    elif y >= 0.35 and x < -0.45:
      reg = "frontal_left"
    elif y >= 0.35 and x > 0.45:
      reg = "frontal_right"
    elif -0.15 <= y < 0.35 and abs(x) <= 0.45:
      reg = "central"
    elif -0.15 <= y < 0.35 and x < -0.45:
      reg = "temporal_left"
    elif -0.15 <= y < 0.35 and x > 0.45:
      reg = "temporal_right"
    elif -0.55 <= y < -0.15 and x <= 0.0:
      reg = "parietal_left"
    elif -0.55 <= y < -0.15 and x > 0.0:
      reg = "parietal_right"
    else:
      reg = "occipital"
    regions[reg].append(idx)
  return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
PHY_REGIONS = build_phy_10_regions()


def precompute_mask_bank(num_masks=1024, device=DEVICE):
  masks = torch.zeros(num_masks, NUM_CHANNELS, dtype=torch.bool, device=device)
  region_keys = list(PHY_REGIONS.keys())
  for b in range(num_masks):
    if random.random() < 0.60:
      k = random.choice([3, 4])
      for rk in random.sample(region_keys, k):
        masks[b, torch.tensor(PHY_REGIONS[rk], device=device)] = True
    else:
      for _, ch_indices in PHY_REGIONS.items():
        n_ch = len(ch_indices)
        if n_ch <= 1:
          continue
        perm = torch.randperm(n_ch, device=device)
        mask_count = max(1, n_ch - 2)
        masks[b, torch.tensor(ch_indices[perm[:mask_count].cpu().numpy()], device=device)] = True
  return masks


MASK_BANK = precompute_mask_bank(1024, DEVICE)


def sample_phy_mask_fast(batch_size, seq_len=WINDOW_LENGTH):
  idx = torch.randint(0, len(MASK_BANK), (batch_size,), device=DEVICE)
  base_mask = MASK_BANK[idx]
  return base_mask.unsqueeze(1).expand(-1, seq_len, -1).reshape(-1, NUM_CHANNELS)


# =====================================================================
# 3. REAL DATA EXTRACTION (4-SECOND NON-OVERLAPPING EPOCHS)
# =====================================================================
def compute_seed_features_4s(raw_trial_eeg, fs=FS, window_sec=EPOCH_SEC, eps=1e-10):
  num_channels, num_samples = raw_trial_eeg.shape
  samples_per_win = int(window_sec * fs)
  num_windows = num_samples // samples_per_win
  if num_windows == 0:
    return None

  truncated = raw_trial_eeg[:, : num_windows * samples_per_win]
  chunks = truncated.reshape(num_channels, num_windows, samples_per_win)

  hann = np.hanning(samples_per_win).astype(np.float32)
  windowed = chunks * hann[None, None, :]
  fft_vals = np.fft.rfft(windowed, n=samples_per_win, axis=-1)
  hann_power = np.sum(hann**2)
  psd = (np.abs(fft_vals) ** 2) / (fs * hann_power)
  freq_bins = np.fft.rfftfreq(samples_per_win, d=1.0 / fs)

  log_psd_bands, de_bands = [], []
  for _, (low_f, high_f) in FREQUENCY_BANDS.items():
    idx = np.where((freq_bins >= low_f) & (freq_bins <= high_f))[0]
    band_power = np.maximum(np.mean(psd[:, :, idx], axis=-1), eps)
    log_psd_bands.append(np.log(band_power).T)
    de_bands.append((0.5 * np.log(2.0 * np.pi * np.e * band_power)).T)

  return np.concatenate(
      [np.stack(log_psd_bands, axis=-1), np.stack(de_bands, axis=-1)], axis=-1
  )


def load_real_seed_dataset():
  if os.path.exists(CACHE_FILE):
    print(f"  [+] Loading cached SEED dataset from: {CACHE_FILE}")
    d = np.load(CACHE_FILE)
    return d["X"], d["y"], d["subs"], d["sess"], d["tri"]

  eeg_dir = None
  for p in DATA_SEARCH_PATHS:
    if os.path.exists(p) and any(f.endswith(".mat") for f in os.listdir(p)):
      eeg_dir = p
      break
    if os.path.exists(os.path.join(p, "Preprocessed_EEG")):
      eeg_dir = os.path.join(p, "Preprocessed_EEG")
      break

  if eeg_dir is not None:
    print(f"  [+] Extracting 4s SEED features from: {eeg_dir}")
    label_file = os.path.join(eeg_dir, "label.mat")
    if not os.path.exists(label_file):
      label_file = os.path.join(os.path.dirname(eeg_dir), "label.mat")
    if os.path.exists(label_file):
      lbl_mat = sio.loadmat(label_file)
      raw_lbl = (
          lbl_mat["label"].flatten()
          if "label" in lbl_mat
          else lbl_mat["labels"].flatten()
      )
      trial_labels = (raw_lbl + 1).astype(np.int64)
    else:
      trial_labels = np.array(SEED_TRIAL_LABELS, dtype=np.int64)

    files_by_subject = {}
    pattern = re.compile(r"^(\d+)_(\d+)\.mat$")
    for f in sorted(os.listdir(eeg_dir)):
      m = pattern.match(f)
      if m:
        files_by_subject.setdefault(int(m.group(1)), []).append((m.group(2), f))

    all_feat, all_y, all_sub, all_ses, all_tri = [], [], [], [], []
    for sub_id in sorted(files_by_subject.keys()):
      for sess_idx, (_, fname) in enumerate(
          sorted(files_by_subject[sub_id], key=lambda x: x[0]), 1
      ):
        mat = sio.loadmat(os.path.join(eeg_dir, fname))
        t_keys = {
            int(re.search(r"eeg(\d+)$", k, re.I).group(1)): k
            for k in mat
            if re.search(r"eeg(\d+)$", k, re.I)
        }

        for t_num in range(1, TRIALS_PER_SESSION + 1):
          if t_num not in t_keys:
            continue
          feats = compute_seed_features_4s(mat[t_keys[t_num]].astype(np.float32))
          if feats is None:
            continue
          n = feats.shape[0]
          all_feat.append(feats)
          all_y.append(np.full(n, trial_labels[t_num - 1], dtype=np.int64))
          all_sub.append(np.full(n, sub_id, dtype=np.int64))
          all_ses.append(np.full(n, sess_idx, dtype=np.int64))
          all_tri.append(np.full(n, t_num, dtype=np.int64))

    if len(all_feat) > 0:
      X = np.concatenate(all_feat, axis=0)
      y = np.concatenate(all_y, axis=0)
      subs = np.concatenate(all_sub, axis=0)
      sess = np.concatenate(all_ses, axis=0)
      tri = np.concatenate(all_tri, axis=0)
      np.savez_compressed(CACHE_FILE, X=X, y=y, subs=subs, sess=sess, tri=tri)
      return X, y, subs, sess, tri

  raise FileNotFoundError("Could not locate SEED raw .mat files. Check dataset mount.")


def normalize_subject_session_independent(x, subs, sess):
  """Strict within-subject-session z-score normalization (zero cross-leakage)."""
  xn = x.copy()
  keys = subs * 100 + sess
  for k in np.unique(keys):
    m = keys == k
    blk = xn[m]
    mu = blk.mean(axis=0, keepdims=True)
    sd = blk.std(axis=0, keepdims=True) + 1e-6
    xn[m] = (blk - mu) / sd
  return xn


def build_sequence_framing(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
  keys = subs * 1000000 + sess * 10000 + tri
  seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []
  order = np.argsort(keys, kind="stable")
  for k in np.unique(keys):
    rows = order[keys[order] == k]
    if rows.shape[0] < seq_len:
      continue
    for start in range(0, rows.shape[0] - seq_len + 1, stride):
      win = rows[start : start + seq_len]
      seqs.append(win)
      labs.append(y[win[0]])
      s_sub.append(subs[win[0]])
      s_ses.append(sess[win[0]])
      s_tri.append(tri[win[0]])
  return (
      np.asarray(seqs, dtype=np.int64),
      np.asarray(labs, dtype=np.int64),
      np.asarray(s_sub, dtype=np.int64),
      np.asarray(s_ses, dtype=np.int64),
      np.asarray(s_tri, dtype=np.int64),
  )


# =====================================================================
# 4. NEUROSCIENCE-INFORMED SPATIAL STMAE
# =====================================================================
class ScalpPositionalEncoding(nn.Module):

  def __init__(self, coords, d_model):
    super().__init__()
    self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
    self.mlp = nn.Sequential(
        nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model)
    )

  def forward(self, x):
    return x + self.mlp(self.coords).unsqueeze(0)


class SpatialSTMAE(nn.Module):

  def __init__(self, in_dim=RAW_DIM, d_model=64, latent_dim=32):
    super().__init__()
    self.latent_dim = latent_dim
    self.proj = nn.Linear(in_dim, d_model)
    self.pos_enc = ScalpPositionalEncoding(SCALP_COORDS, d_model)
    self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
    nn.init.normal_(self.mask_token, std=0.02)

    enc_layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=4,
        dim_feedforward=d_model * 2,
        dropout=0.1,
        batch_first=True,
        norm_first=True,
        activation="gelu",
    )
    self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
    self.to_latent = nn.Sequential(
        nn.Linear(d_model, latent_dim), nn.LayerNorm(latent_dim)
    )
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, d_model), nn.GELU(), nn.Linear(d_model, in_dim)
    )

  def encode(self, x, mask=None):
    h = self.proj(x)
    if mask is not None:
      h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
    h = self.pos_enc(h)
    h = self.encoder(h)
    return self.to_latent(h)

  def forward_pretrain(self, x, mask):
    z = self.encode(x, mask)
    return self.decoder(z)

  def extract_latents(self, x, mask=None):
    B, T, C, F_dim = x.shape
    flat = x.reshape(B * T, C, F_dim)
    mask_flat = mask.expand(B * T, C) if mask is not None else None
    z = self.encode(flat, mask=mask_flat)
    return z.reshape(B, T, C * self.latent_dim)


def pretrain_stmae_fold_internal_amp_fast(
    x_pt,
    pt_seq_indices,
    train_rows_pt,
    in_dim=RAW_DIM,
    fold_name="Fold",
    epochs=FOLD_PRETRAIN_EPOCHS,
):
  """Fast Strict Inductive Pretraining on 14 training subjects with zero test-subject leakage."""
  stmae = SpatialSTMAE(in_dim=in_dim).to(DEVICE)
  opt = torch.optim.AdamW(stmae.parameters(), lr=PRETRAIN_LR, weight_decay=1e-4)
  sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
  scaler = torch.amp.GradScaler("cuda")

  tr_seqs = pt_seq_indices[train_rows_pt]
  n_seq = len(tr_seqs)
  stmae.train()

  print(f"    [+] Fast Pretraining STMAE on 14 Train Subjects ({epochs} epochs, AMP)...")
  for ep in range(1, epochs + 1):
    perm = torch.randperm(n_seq, device=DEVICE)
    tot_loss, steps = 0.0, 0
    for i in range(0, n_seq, PRETRAIN_BATCH_SIZE):
      b_idx = perm[i : i + PRETRAIN_BATCH_SIZE]
      xb = x_pt[tr_seqs[b_idx]]
      xb_flat = xb.reshape(-1, NUM_CHANNELS, in_dim)

      mask = sample_phy_mask_fast(xb.size(0), WINDOW_LENGTH)

      opt.zero_grad()
      with torch.amp.autocast("cuda"):
        recon = stmae.forward_pretrain(xb_flat, mask)
        loss = F.mse_loss(recon[mask], xb_flat[mask])

      scaler.scale(loss).backward()
      scaler.unscale_(opt)
      nn.utils.clip_grad_norm_(stmae.parameters(), GRADIENT_CLIP)
      scaler.step(opt)
      scaler.update()

      tot_loss += loss.item()
      steps += 1

    sched.step()
    if ep == 1 or ep % 5 == 0 or ep == epochs:
      avg_mse = tot_loss / max(steps, 1)
      print(f"      [{fold_name} - STMAE Fast] Epoch {ep:02d}/{epochs:02d} | Recon MSE: {avg_mse:.5f}")

  return stmae


# =====================================================================
# 5. TEMPORAL MSC-TIMESNET & CLASSIFIER
# =====================================================================
class MultiScaleConvBlock(nn.Module):

  def __init__(self, channels):
    super().__init__()
    mid = channels // 4
    self.b1 = nn.Conv2d(channels, mid, kernel_size=1, padding=0)
    self.b3 = nn.Conv2d(channels, mid, kernel_size=3, padding=1)
    self.b5 = nn.Conv2d(channels, mid, kernel_size=5, padding=2)
    self.bp = nn.Sequential(
        nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
        nn.Conv2d(channels, mid, kernel_size=1),
    )
    self.fuse = nn.Conv2d(mid * 4, channels, kernel_size=1)
    self.bn = nn.BatchNorm2d(channels)
    self.act = nn.GELU()

  def forward(self, x):
    res = x
    out = torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)
    out = self.bn(self.fuse(out))
    return self.act(out + res)


class MSCTimesNet(nn.Module):

  def __init__(self, d_model=128, top_k=2):
    super().__init__()
    self.d_model = d_model
    self.top_k = top_k
    self.conv = MultiScaleConvBlock(d_model)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x):
    B, T, D = x.shape
    fft_energy = torch.abs(torch.fft.rfft(x, dim=1)).mean(dim=(0, 2))
    fft_energy[0] = 0.0

    _, topk_indices = torch.topk(fft_energy, self.top_k)
    periods = [
        max(2, int(round(T / idx.item()))) if idx.item() > 0 else T
        for idx in topk_indices
    ]

    period_outputs = []
    for p in periods:
      pad_len = (p - (T % p)) % p
      x_pad = F.pad(x, (0, 0, 0, pad_len)) if pad_len > 0 else x
      T_pad = x_pad.shape[1]

      x_2d = x_pad.reshape(B, T_pad // p, p, D).permute(0, 3, 1, 2)
      out_2d = self.conv(x_2d)
      out_1d = out_2d.permute(0, 2, 3, 1).reshape(B, T_pad, D)[:, :T, :]
      period_outputs.append(out_1d)

    agg = torch.stack(period_outputs, dim=0).mean(dim=0)
    return self.norm(x + agg)


class FullEmotionModel(nn.Module):

  def __init__(self, fold_stmae, num_classes=NUM_CLASSES):
    super().__init__()
    self.stmae = fold_stmae
    self.input_embed = nn.Sequential(
        nn.Linear(NUM_CHANNELS * 32, 128), nn.LayerNorm(128)
    )
    self.timesnet = MSCTimesNet(d_model=128, top_k=2)
    self.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.GELU(),
        nn.Linear(64, num_classes),
    )

  def forward(self, x, mask=None):
    h = self.stmae.extract_latents(x, mask=mask)
    h = self.input_embed(h)
    h = self.timesnet(h)
    return self.classifier(h.mean(dim=1))


# =====================================================================
# 6. FAST SUPERVISED FINE-TUNING & FULL INFERENCE
# =====================================================================
def fit_fold_classifier_amp(model, x_pt, seq_indices, train_rows, y_all, groups):
  gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
  tr_local, va_local = next(
      gss.split(train_rows, y_all[train_rows], groups[train_rows])
  )

  tr_rows_pt = torch.tensor(train_rows[tr_local], dtype=torch.long, device=DEVICE)
  va_rows_pt = torch.tensor(train_rows[va_local], dtype=torch.long, device=DEVICE)
  labels_pt = torch.tensor(y_all, dtype=torch.long, device=DEVICE)

  opt = torch.optim.AdamW([
      {"params": model.stmae.parameters(), "lr": CLASSIFIER_LR * ENCODER_LR_SCALE},
      {"params": model.input_embed.parameters(), "lr": CLASSIFIER_LR},
      {"params": model.timesnet.parameters(), "lr": CLASSIFIER_LR},
      {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
  ], weight_decay=WEIGHT_DECAY)

  sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLASSIFIER_EPOCHS)
  criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
  scaler = torch.amp.GradScaler("cuda")

  best_f1 = -1.0
  best_weights = None

  for ep in range(1, CLASSIFIER_EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(tr_rows_pt), device=DEVICE)
    for i in range(0, len(tr_rows_pt), CLASSIFIER_BATCH_SIZE):
      b_idx = tr_rows_pt[perm[i : i + CLASSIFIER_BATCH_SIZE]]
      xb = x_pt[seq_indices[b_idx]]
      yb = labels_pt[b_idx]

      if CHANNEL_DROPOUT_RATE > 0:
        keep = (torch.rand(xb.size(0), 1, NUM_CHANNELS, 1, device=DEVICE) > CHANNEL_DROPOUT_RATE).float()
        xb = xb * keep

      opt.zero_grad()
      with torch.amp.autocast("cuda"):
        logits = model(xb)
        loss = criterion(logits, yb)

      scaler.scale(loss).backward()
      scaler.unscale_(opt)
      nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
      scaler.step(opt)
      scaler.update()

    sched.step()

    if ep % 3 == 0 or ep == CLASSIFIER_EPOCHS:
      model.eval()
      with torch.no_grad(), torch.amp.autocast("cuda"):
        xb_va = x_pt[seq_indices[va_rows_pt]]
        preds_va = model(xb_va).argmax(dim=1).cpu().numpy()
        f1 = f1_score(y_all[train_rows[va_local]], preds_va, average="macro", zero_division=0)
        if f1 > best_f1:
          best_f1 = f1
          best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

  if best_weights is not None:
    model.load_state_dict(best_weights)
  return model


@torch.no_grad()
def evaluate_test_subject_full_metrics(model, x_pt, seq_indices, test_rows, y_all, s_ses, s_tri):
  model.eval()
  te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)
  probs = []
  for i in range(0, len(test_rows), CLASSIFIER_BATCH_SIZE):
    batch = x_pt[seq_indices[te_rows_pt[i : i + CLASSIFIER_BATCH_SIZE]]]
    with torch.amp.autocast("cuda"):
      logits = model(batch)
      probs.append(F.softmax(logits, dim=1).cpu().numpy())

  probs = np.concatenate(probs, axis=0)
  y_true = y_all[test_rows]
  y_pred = probs.argmax(axis=1)

  # 1. Window-Wise Metrics
  w_acc = accuracy_score(y_true, y_pred)
  w_bacc = balanced_accuracy_score(y_true, y_pred)
  w_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
  w_weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

  # 2. Multi-Session Isolated Consensus: 15 trials x 3 sessions = Exactly 45 Trials
  unique_trial_keys = s_ses[test_rows] * 100 + s_tri[test_rows]
  yt_trial, yp_trial = [], []
  for t_k in np.unique(unique_trial_keys):
    idx = np.where(unique_trial_keys == t_k)[0]
    yt_trial.append(y_true[idx[0]])
    yp_trial.append(probs[idx].mean(axis=0).argmax())

  yt_trial, yp_trial = np.array(yt_trial), np.array(yp_trial)
  t_acc = accuracy_score(yt_trial, yp_trial)
  t_bacc = balanced_accuracy_score(yt_trial, yp_trial)
  t_macro_f1 = f1_score(yt_trial, yp_trial, average="macro", zero_division=0)
  t_weighted_f1 = f1_score(yt_trial, yp_trial, average="weighted", zero_division=0)

  return {
      "win_acc": w_acc,
      "win_bacc": w_bacc,
      "win_macro_f1": w_macro_f1,
      "win_weighted_f1": w_weighted_f1,
      "trial_acc": t_acc,
      "trial_bacc": t_bacc,
      "trial_macro_f1": t_macro_f1,
      "trial_weighted_f1": t_weighted_f1,
      "num_windows": len(test_rows),
      "num_trials": len(yt_trial),
  }


# =====================================================================
# 7. EMOTION-SPECIFIC EEG CHANNEL ACCURACY EVALUATOR
# =====================================================================
@torch.no_grad()
def evaluate_channel_accuracies(
    model, x_pt, seq_indices, y_all, channel_names=CHANNEL_NAMES, top_k=10
):
  model.eval()
  n_samples = len(seq_indices)
  num_channels = len(channel_names)
  emotion_names = {0: "Negative", 1: "Neutral", 2: "Positive"}

  def predict_masked(mask_1d):
    all_probs = []
    mask_2d = mask_1d.unsqueeze(0).to(DEVICE)
    for i in range(0, n_samples, CLASSIFIER_BATCH_SIZE):
      xb = x_pt[seq_indices[i : i + CLASSIFIER_BATCH_SIZE]]
      with torch.amp.autocast("cuda"):
        logits = model(xb, mask=mask_2d)
        all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(all_probs, axis=0).argmax(axis=1)

  # Full-montage baseline
  full_mask = torch.zeros(num_channels, dtype=torch.bool, device=DEVICE)
  base_preds = predict_masked(full_mask)
  base_acc = accuracy_score(y_all, base_preds)
  base_mac_f1 = f1_score(y_all, base_preds, average="macro", zero_division=0)
  base_wt_f1 = f1_score(y_all, base_preds, average="weighted", zero_division=0)

  print("\n" + "=" * 90)
  print(f"CHANNEL ACCURACY ANALYSIS BASELINE (All 62 Channels): Acc={base_acc:.4f} | Macro-F1={base_mac_f1:.4f} | Weighted-F1={base_wt_f1:.4f}")
  print("=" * 90)

  # Standalone single-channel accuracy
  print("  [+] Computing Standalone Accuracies & Per-Emotion Sensitivity for each channel...")
  records = []
  for ch_idx, ch_name in enumerate(channel_names):
    single_mask = torch.ones(num_channels, dtype=torch.bool, device=DEVICE)
    single_mask[ch_idx] = False

    preds = predict_masked(single_mask)
    acc = accuracy_score(y_all, preds)
    mac_f1 = f1_score(y_all, preds, average="macro", zero_division=0)
    wt_f1 = f1_score(y_all, preds, average="weighted", zero_division=0)

    class_accs = {}
    for c in [0, 1, 2]:
      c_mask = (y_all == c)
      c_correct = (preds[c_mask] == c).sum()
      c_total = np.sum(c_mask)
      class_accs[emotion_names[c]] = c_correct / c_total if c_total > 0 else 0.0

    records.append({
        "channel": ch_name,
        "standalone_acc": acc,
        "macro_f1": mac_f1,
        "weighted_f1": wt_f1,
        "neg_acc": class_accs["Negative"],
        "neu_acc": class_accs["Neutral"],
        "pos_acc": class_accs["Positive"],
    })

  df_channels = pd.DataFrame(records)
  top_overall = df_channels.sort_values(by="standalone_acc", ascending=False).reset_index(drop=True)

  print(f"\nTOP-{top_k} CHANNELS BY STANDALONE ACCURACY:")
  print("-" * 90)
  print(top_overall.head(top_k).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  print("\nTOP SPECIALIZED CHANNELS PER EMOTION:")
  print("-" * 90)
  for col, name in [("neg_acc", "Negative"), ("neu_acc", "Neutral"), ("pos_acc", "Positive")]:
    best_c = df_channels.sort_values(by=col, ascending=False).iloc[0]
    print(f"  * {name:<8} Emotion -> Channel: {best_c['channel']:<4} | Sensitivity: {best_c[col]:.4f} (Overall Acc: {best_c['standalone_acc']:.4f})")

  # Cumulative Top-K Subsets
  print("\nCUMULATIVE SUBSET ACCURACIES (Top-K Channels Retained):")
  print("-" * 90)
  ranked_names = list(top_overall["channel"])
  subset_records = []
  for k in [1, 3, 5, 10, 20]:
    if k > num_channels:
      continue
    top_k_names = ranked_names[:k]
    top_k_idx = [channel_names.index(n) for n in top_k_names]

    k_mask = torch.ones(num_channels, dtype=torch.bool, device=DEVICE)
    k_mask[top_k_idx] = False

    k_preds = predict_masked(k_mask)
    subset_records.append({
        "configuration": f"Top-{k:<2} Channels",
        "accuracy": accuracy_score(y_all, k_preds),
        "macro_f1": f1_score(y_all, k_preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_all, k_preds, average="weighted", zero_division=0),
        "channels_used": ", ".join(top_k_names[:5]) + ("..." if k > 5 else ""),
    })

  df_subsets = pd.DataFrame(subset_records)
  print(df_subsets.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
  print("=" * 90)

  return df_channels, df_subsets


# =====================================================================
# 8. MAIN STRICT INDUCTIVE PIPELINE EXECUTION
# =====================================================================
def main():
  seed_everything(RANDOM_SEED)
  t0 = time.time()
  print("=" * 90)
  print("STRICT INDUCTIVE LOSO PIPELINE (SEED 3-CLASS, FAST RECONSTRUCTION)")
  print(f"Device: {DEVICE} | Channels: {NUM_CHANNELS} | Context: 40s (T=10)")
  print("=" * 90)

  # 1. Ingest Data & Standardize independently per session
  x_raw, y, subs, sess, tri = load_real_seed_dataset()
  in_dim = x_raw.shape[-1]
  x_norm = normalize_subject_session_independent(x_raw, subs, sess)
  x_pt = torch.tensor(x_norm, dtype=torch.float32, device=DEVICE)

  # 2. Build Dense Framing (Stride=1) for Fine-Tuning & Evaluation
  seq_idx, y_seq, s_sub, s_ses, s_tri = build_sequence_framing(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE)
  seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
  groups = s_sub * 1000000 + s_ses * 10000 + s_tri
  unique_subs = np.unique(s_sub)

  # 3. Build Non-Overlapping Framing (Stride=10) Exclusively for Fast STMAE Pretraining
  pt_seq_idx, _, pt_s_sub, _, _ = build_sequence_framing(
      y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=PRETRAIN_STRIDE
  )
  pt_seq_idx_pt = torch.tensor(pt_seq_idx, dtype=torch.long, device=DEVICE)

  print(f"  [+] Dense Evaluation Windows (Stride=1): {len(seq_idx_pt)} across {len(unique_subs)} Subjects.")
  print(f"  [+] Fast Pretraining Windows (Stride=10): {len(pt_seq_idx_pt)} across {len(unique_subs)} Subjects.")

  loso_records = []
  last_trained_model = None
  last_test_indices = None
  last_test_labels = None

  print("\n" + "-" * 90)
  print("STARTING STRICT INDUCTIVE LEAVE-ONE-SUBJECT-OUT (15 SUBJECTS)")
  print("-" * 90)

  for target_sub in unique_subs:
    t_fold = time.time()
    tr_rows = np.where(s_sub != target_sub)[0]
    te_rows = np.where(s_sub == target_sub)[0]
    tr_rows_pt = np.where(pt_s_sub != target_sub)[0]

    fold_tag = f"Subject_{target_sub:02d}"
    print(f"\n>>> Fold: {fold_tag} (Training on 14 Subjects, Testing on {fold_tag}) <<<")

    # Step 1: Fast STMAE Pretraining on 14 training subjects (~35s with AMP)
    fold_stmae = pretrain_stmae_fold_internal_amp_fast(
        x_pt,
        pt_seq_idx_pt,
        tr_rows_pt,
        in_dim=in_dim,
        fold_name=fold_tag,
        epochs=FOLD_PRETRAIN_EPOCHS,
    )

    # Step 2: Supervised Fine-Tuning
    print(f"    [+] Fine-tuning FullEmotionModel on {fold_tag} training partition...")
    model = FullEmotionModel(fold_stmae, num_classes=NUM_CLASSES).to(DEVICE)
    model = fit_fold_classifier_amp(model, x_pt, seq_idx_pt, tr_rows, y_seq, groups)

    # Step 3: Blind Test Inference
    metrics = evaluate_test_subject_full_metrics(model, x_pt, seq_idx_pt, te_rows, y_seq, s_ses, s_tri)
    metrics["subject"] = fold_tag
    loso_records.append(metrics)

    last_trained_model = model
    last_test_indices = seq_idx_pt[te_rows]
    last_test_labels = y_seq[te_rows]

    print(
        f"  -> Finished {fold_tag} | "
        f"WIN: acc={metrics['win_acc']:.4f} bacc={metrics['win_bacc']:.4f} macF1={metrics['win_macro_f1']:.4f} wtF1={metrics['win_weighted_f1']:.4f} | "
        f"TRIAL: acc={metrics['trial_acc']:.4f} bacc={metrics['trial_bacc']:.4f} macF1={metrics['trial_macro_f1']:.4f} wtF1={metrics['trial_weighted_f1']:.4f} "
        f"({metrics['num_trials']}t / {(time.time() - t_fold) / 60.0:.2f} min)"
    )

  # Grand Summary
  df_loso = pd.DataFrame(loso_records)
  df_loso.to_csv(os.path.join(OUTPUT_DIR, "results_strict_loso_metrics.csv"), index=False)

  print("\n" + "=" * 90)
  print("STRICT INDUCTIVE LOSO GRAND SUMMARY (15 SUBJECTS):")
  print("=" * 90)
  metric_cols = [
      "win_acc", "win_bacc", "win_macro_f1", "win_weighted_f1",
      "trial_acc", "trial_bacc", "trial_macro_f1", "trial_weighted_f1"
  ]
  summary_df = pd.DataFrame({
      "Metric": metric_cols,
      "Mean": [df_loso[m].mean() for m in metric_cols],
      "Std": [df_loso[m].std() for m in metric_cols],
  })
  print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  # Step 4: Emotion-Specific Channel Accuracies & Top-K Subsets
  if last_trained_model is not None:
    df_channels, df_subsets = evaluate_channel_accuracies(
        model=last_trained_model,
        x_pt=x_pt,
        seq_indices=last_test_indices,
        y_all=last_test_labels,
        channel_names=CHANNEL_NAMES,
        top_k=10,
    )
    df_channels.to_csv(os.path.join(OUTPUT_DIR, "results_channel_standalone_accuracies.csv"), index=False)
    df_subsets.to_csv(os.path.join(OUTPUT_DIR, "results_channel_subsets_accuracies.csv"), index=False)

  print(f"\n[+] Full Strict Inductive Pipeline Completed in {(time.time() - t0) / 60.0:.2f} minutes.")


if __name__ == "__main__":
  main()

STRICT INDUCTIVE LOSO PIPELINE (SEED 3-CLASS, FAST RECONSTRUCTION)
Device: cuda | Channels: 62 | Context: 40s (T=10)
  [+] Extracting 4s SEED features from: /kaggle/input/datasets/yunzinan/seed-preprocessed/Preprocessed_EEG
  [+] Dense Evaluation Windows (Stride=1): 31815 across 15 Subjects.
  [+] Fast Pretraining Windows (Stride=10): 3330 across 15 Subjects.

------------------------------------------------------------------------------------------
STARTING STRICT INDUCTIVE LEAVE-ONE-SUBJECT-OUT (15 SUBJECTS)
------------------------------------------------------------------------------------------

>>> Fold: Subject_01 (Training on 14 Subjects, Testing on Subject_01) <<<
    [+] Fast Pretraining STMAE on 14 Train Subjects (50 epochs, AMP)...
      [Subject_01 - STMAE Fast] Epoch 01/50 | Recon MSE: 0.85199
      [Subject_01 - STMAE Fast] Epoch 05/50 | Recon MSE: 0.52521
      [Subject_01 - STMAE Fast] Epoch 10/50 | Recon MSE: 0.40322
      [Subject_01 - STMAE Fast] Epoch 15/50 | Recon